# Dashboard SISCAN — execução no Databricks

Inicia o Streamlit para desenvolvimento e testes, sem modificar tabelas Medallion ou o deploy público.

Execute a partir de `notebooks/` no repositório Git, em compute clássico dedicado (Databricks Runtime 14+). Configure as dependências e credenciais conforme `src/app/README.md`, utilizando segredos ou variáveis de ambiente. Não inclua credenciais no notebook.

O acesso depende do proxy autenticado e das permissões do workspace. O serviço permanece disponível enquanto o processo e o compute estiverem ativos.

In [0]:
%pip install -r ../src/app/requirements.txt

In [0]:
import os
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

PORT = 8501
HOST = "0.0.0.0"
STARTUP_TIMEOUT = 60
STOP_TIMEOUT = 10
HEALTH_TIMEOUT = 2
POLL_INTERVAL = 0.5


def localizar_app() -> Path:
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        app = root / "src" / "app"
        if (app / "main.py").is_file() and (app / "requirements.txt").is_file():
            return app
    raise FileNotFoundError("Execute o notebook no repositório siscan-data-analytics.")


APP_DIR = localizar_app()


def porta_disponivel(porta: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind((HOST, porta))
            return True
        except OSError:
            return False


def servidor_pronto(porta: int) -> bool:
    url = f"http://127.0.0.1:{porta}/_stcore/health"
    try:
        with urlopen(url, timeout=HEALTH_TIMEOUT) as response:
            return response.status == 200 and response.read().strip() == b"ok"
    except (URLError, TimeoutError, OSError):
        return False


def encerrar_processo(processo: subprocess.Popen) -> None:
    if processo.poll() is not None:
        return
    processo.terminate()
    try:
        processo.wait(timeout=STOP_TIMEOUT)
    except subprocess.TimeoutExpired:
        processo.kill()
        processo.wait(timeout=STOP_TIMEOUT)


def parar_streamlit() -> None:
    processo = globals().get("streamlit_process")
    if processo is not None:
        encerrar_processo(processo)


def aguardar_inicializacao(processo: subprocess.Popen, porta: int) -> None:
    deadline = time.monotonic() + STARTUP_TIMEOUT
    while time.monotonic() < deadline:
        if processo.poll() is not None:
            raise RuntimeError(f"Streamlit encerrou com código {processo.returncode}.")
        if servidor_pronto(porta) and processo.poll() is None:
            return
        time.sleep(POLL_INTERVAL)
    raise TimeoutError(f"Streamlit não respondeu em {STARTUP_TIMEOUT} segundos.")


def iniciar_streamlit(porta: int = PORT) -> tuple[subprocess.Popen, Path]:
    if not 1 <= porta <= 65535:
        raise ValueError("Porta inválida: utilize um número entre 1 e 65535.")

    # Nunca encerra serviços que não foram iniciados por este notebook.
    if not porta_disponivel(porta):
        raise RuntimeError(f"Porta {porta} ocupada. Altere a porta ou encerre o serviço responsável.")

    command = [
        sys.executable, "-m", "streamlit", "run", str(APP_DIR / "main.py"),
        "--server.port", str(porta),
        "--server.address", HOST,
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
    ]
    log_path = Path(tempfile.mkdtemp(prefix="siscan_streamlit_")) / "streamlit.log"

    with log_path.open("w", encoding="utf-8") as log:
        processo = subprocess.Popen(
            command,
            cwd=str(APP_DIR),
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    try:
        aguardar_inicializacao(processo, porta)
    except BaseException as error:
        # Inclui interrupções manuais durante a inicialização.
        encerrar_processo(processo)
        if isinstance(error, (RuntimeError, TimeoutError)):
            raise RuntimeError(f"{error} Consulte o log local: {log_path}") from error
        raise

    return processo, log_path

In [0]:
parar_streamlit()
streamlit_process, streamlit_log = iniciar_streamlit(PORT)
streamlit_port = PORT

print(f"Streamlit disponível | porta {streamlit_port} | PID {streamlit_process.pid}")
print(f"Log local: {streamlit_log}")

In [0]:
from html import escape
from urllib.parse import urlsplit

from IPython.display import HTML, display


def config_spark(key: str) -> str:
    try:
        return spark.conf.get(key, "") or ""
    except Exception:
        return ""


def url_proxy_classico(porta: int) -> str | None:
    host = config_spark("spark.databricks.workspaceUrl")
    workspace = config_spark("spark.databricks.clusterUsageTags.clusterOwnerOrgId")
    cluster = config_spark("spark.databricks.clusterUsageTags.clusterId")
    if not host or not workspace or not cluster:
        return None

    base = host if host.startswith("https://") else f"https://{host}"
    return f"{base.rstrip('/')}/driver-proxy/o/{workspace}/{cluster}/{porta}/"


def validar_url(url: str) -> str:
    parsed = urlsplit(url)
    if (
        parsed.scheme != "https"
        or not parsed.hostname
        or parsed.username is not None
        or parsed.password is not None
        or any(char.isspace() for char in url)
    ):
        raise ValueError("Informe uma URL HTTPS autenticada, sem credenciais embutidas.")
    return url


def publicar_link(porta: int) -> None:
    # O localhost do navegador não corresponde ao compute remoto.
    url = os.getenv("SISCAN_STREAMLIT_URL") or url_proxy_classico(porta)
    if not url:
        print(f"Encaminhe a porta {porta} pelo recurso autorizado do Databricks.")
        print("Configure SISCAN_STREAMLIT_URL se houver uma URL autenticada disponível.")
        return

    url = validar_url(url)
    html = (
        f'<a href="{escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">'
        "Abrir dashboard SISCAN</a>"
    )
    if "displayHTML" in globals():
        displayHTML(html)
    else:
        display(HTML(html))


processo = globals().get("streamlit_process")
porta = globals().get("streamlit_port")
if processo is None or porta is None or processo.poll() is not None or not servidor_pronto(porta):
    raise RuntimeError("Servidor indisponível. Execute a célula de inicialização.")

publicar_link(porta)

## Encerrar ou reiniciar

Execute `parar_streamlit()` em uma nova célula para finalizar apenas o processo desta sessão. Para reiniciar, execute novamente a célula de inicialização; não é necessário reinstalar as dependências.

Se a instalação reiniciar a sessão Python, execute novamente as células seguintes. Uma porta ocupada por outra sessão gera erro, sem encerrar processos desconhecidos.

O endpoint de saúde verifica o servidor Streamlit, não a conexão ao SQL Warehouse. Confirme as consultas pela interface. Em serverless, o proxy e o diretório de trabalho podem diferir; utilize o encaminhamento autorizado do workspace.